# Lab qc_04 — Quantum Fourier transform and phase estimation

**Skills:** `qc.qft`, `qc.phase_estimation` · ~90 min · CPU

### Goals
1. Write the QFT matrix and a QFT **circuit** from $H$, controlled-phase and SWAP gates; verify against Qiskit's `QFTGate`.
2. Build quantum phase estimation (QPE) for $U = P(\theta) = \mathrm{diag}(1, e^{i\theta})$ — exact for the $T$ gate, approximate for $\varphi = 1/3$.

### Theory recap
$$\mathrm{QFT}\lvert j\rangle = \frac{1}{\sqrt N}\sum_{k=0}^{N-1} e^{2\pi i jk/N}\lvert k\rangle = \bigotimes_{l}\frac{\lvert0\rangle + e^{2\pi i j/2^{l}}\lvert1\rangle}{\sqrt2}.$$
The product form gives the circuit: for each qubit (most significant first) apply $H$ and controlled phases $\pi/2^{d}$ from the less-significant qubits, then reverse the qubit order with $\lfloor n/2\rfloor$ SWAPs: $n(n-1)/2$ controlled phases in total ($O(n^2)$ vs $O(n2^n)$ for the FFT).
Integers $j,k$ use **Qiskit's little-endian** indexing (qubit 0 = least significant bit).

**QPE.** With $U\lvert u\rangle = e^{2\pi i\varphi}\lvert u\rangle$: put $t$ counting qubits in $\lvert+\rangle$, let counting qubit $k$ control $U^{2^k}$ — phase kickback gives $\frac{1}{\sqrt{2^t}}\sum_j e^{2\pi i\varphi j}\lvert j\rangle = \mathrm{QFT}\lvert 2^t\varphi\rangle$. The inverse QFT then reveals $m = 2^t\varphi$ exactly if it is an integer; otherwise
$$P(m) = \Big|\frac{1}{2^t}\sum_{k} e^{2\pi i k(\varphi - m/2^t)}\Big|^2,\quad P(\text{nearest}) \ge 4/\pi^2.$$
For $P(\theta)$, $U^{2^k} = P(2^k\theta)$ — a single `cp(theta * 2**k, k, t)`.

### Read
- Shor, 8.370/18.435 lecture notes — https://math.mit.edu/~shor/435-LN/
- IBM Quantum Learning, *Fundamentals of Quantum Algorithms* (phase estimation) — https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms
- Shor 1994/95 — https://arxiv.org/abs/quant-ph/9508027

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate
from qiskit.quantum_info import Operator
from qiskit_aer import AerSimulator

## 1. QFT matrix and circuit

In [ ]:
def qft_matrix(n):
    # TODO: F[k, j] = exp(2πi jk/N)/sqrt(N)
    raise NotImplementedError

def qft_circuit(n, do_swaps=True):
    # TODO: for j = n-1 down to 0: H(j); for k = j-1 down to 0: cp(pi / 2**(j-k), k, j). Then swaps.
    raise NotImplementedError

def inverse_qft_circuit(n):
    # TODO (hint: QuantumCircuit.inverse())
    raise NotImplementedError

In [ ]:
n = 3
ref = QuantumCircuit(n); ref.append(QFTGate(n), range(n))
print(qft_circuit(n).draw())
print("matches QFTGate:", np.allclose(Operator(qft_circuit(n)).data, Operator(ref).data))
print("matches matrix :", np.allclose(Operator(qft_circuit(n)).data, qft_matrix(n)))

## 2. Phase estimation
`qpe_circuit(theta, t)`: $t+1$ qubits, $t$ clbits; target qubit $t$ prepared in $\lvert1\rangle$; counting qubit $k$ → clbit $k$, so the counts bitstring read as a binary integer is $m$. `run_counts(qc, shots, seed)` runs on `AerSimulator`. `estimate_phase(counts, t)` = most frequent $m / 2^t$. `qpe_distribution(phi, t)` = exact $P(m)$ array.

In [ ]:
def qpe_circuit(theta, t):
    # TODO
    raise NotImplementedError

def run_counts(qc, shots=2000, seed=0):
    # TODO
    raise NotImplementedError

def estimate_phase(counts, t):
    # TODO
    raise NotImplementedError

def qpe_distribution(phi, t):
    # TODO
    raise NotImplementedError

## 3. Experiments

In [ ]:
print("T gate, t=3:", run_counts(qpe_circuit(np.pi / 4, 3), shots=500))

t, phi = 5, 1 / 3
counts = run_counts(qpe_circuit(2 * np.pi * phi, t), shots=4000, seed=1)
ms = np.arange(2 ** t)
emp = np.array([counts.get(format(m, f"0{t}b"), 0) for m in ms]) / 4000
plt.bar(ms, emp, alpha=0.6, label="AerSimulator")
plt.plot(ms, qpe_distribution(phi, t), "r.", label="theory")
plt.axvline(phi * 2 ** t, ls=":", c="k"); plt.legend(); plt.xlabel("m"); plt.title("QPE for φ = 1/3, t = 5"); plt.show()
print("estimate:", estimate_phase(counts, t), " true:", phi)

### Reflect (on paper)
1. How many counting qubits are needed to get $\varphi$ to $b$ bits with success probability $\ge 1-\epsilon$? (N&C Eq. 5.35.)
2. In Shor's algorithm, which unitary plays the role of $U$, and why is $\lvert1\rangle$ (mod $N$) a uniform superposition of its eigenvectors?

## Grade
Run all cells above, then this one. It runs the hidden checks and records the result in the Learning Center.

In [ ]:
from lc_client import grade
grade("qc_04_qft_qpe", globals())